# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
### Finding 1

The paper reports that content age is a strong negative signal in its growth prediction analysis, while days visible and recent impressions are among the stronger positive signals.

**My methodology question:** Where does the growth/decline label come from, and is it based on a future outcome window that is kept separate from the features used for prediction? The paper states that the ML analysis is exploratory, so I would want to confirm that the validation design supports the strength of the claim.

### Finding 2

The paper reports that the strongest measured freshness window was 31–90 days, with a 7.88:1 growth-to-decline ratio, and recommends refreshing mature pages before they decay.

**My methodology question:** How is the growth-versus-decline label defined for this comparison, and does the validation design account for differences between clients and time periods? Since the study is observational, I would treat this as a directional association rather than evidence that updating a page directly causes growth.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Split design**: I use a grouped split by client so that the same client cannot appear in both training and test sets. This gives a more honest estimate of how the model may perform on unseen clients. I compare the grouped-split result with the Week-5 result using Average Precision.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score

# Load March and April data
march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

april = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet"
)

# Aggregate March data by client and page
march_agg = (
    march.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_sum_position=("gsc_sum_position", "sum")
    )
)

march_agg["march_position"] = (
    march_agg["march_sum_position"] / march_agg["march_impressions"]
)

# Aggregate April data
april_agg = (
    april.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        april_impressions=("gsc_impressions", "sum")
    )
)

# Keep pages that had March impressions
model_df = march_agg[march_agg["march_impressions"] > 0].copy()

# Add April impressions
model_df = model_df.merge(
    april_agg,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

model_df["april_impressions"] = model_df["april_impressions"].fillna(0)

# Target: impression decline from March to April
model_df["target_decline"] = (
    model_df["april_impressions"] < model_df["march_impressions"]
).astype(int)

# Features used in Week 5
features = [
    "march_impressions",
    "march_clicks",
    "march_position"
]

target = "target_decline"

# Remove invalid values
model_df = model_df.dropna(subset=features + [target])

print("Model rows:", len(model_df))
print("Target distribution:")
print(model_df[target].value_counts())

Model rows: 176738
Target distribution:
target_decline
1    111968
0     64770
Name: count, dtype: int64


In [5]:
# Honest grouped split: clients in train and test are completely separate

X = model_df[features].copy()
y = model_df[target].copy()
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.22,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(model_df.iloc[train_idx]["client_hash_id"])
test_clients = set(model_df.iloc[test_idx]["client_hash_id"])

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Shared clients:", len(train_clients & test_clients))

Train rows: 133491
Test rows: 43247
Train clients: 36
Test clients: 11
Shared clients: 0


In [6]:
model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

test_prob = model.predict_proba(X_test)[:, 1]

honest_ap = average_precision_score(y_test, test_prob)

print("Honest grouped-split Average Precision:", honest_ap)

Honest grouped-split Average Precision: 0.6113944595078553


In [7]:
comparison = pd.DataFrame({
    "Method": [
        "Week-5 baseline",
        "Week-5 Decision Tree",
        "Week-6 Decision Tree - grouped split"
    ],
    "Average Precision": [
        0.448416,
        0.489534,
        honest_ap
    ]
})

comparison

,Method,Average Precision
0,Week-5 baseline,0.448416
1,Week-5 Decision Tree,0.489534
2,Week-6 Decision Tree - grouped split,0.611394


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.